# CSCI4052 GROUP FINAL PROJECT 

MEMBERS: Tony Akinniranye, Lukas Fenkam, Jaathavan Ranjanathan

For our project, we dedided to make a sign language interpreter


In [ ]:
from collections import Counter
from pathlib import Path

from datasets import load_dataset, load_from_disk

In [ ]:
"""# 1. Load the dataset (prefer local full dataset if present)"""

TOP_K = 100
DATASET_NAME = "Voxel51/WLASL"
SPLIT = "train"
LOCAL_DATASET_PATH = Path("wlasl_full_dataset/dataset")

if LOCAL_DATASET_PATH.exists():
    dataset = load_from_disk(str(LOCAL_DATASET_PATH))
    print(f"Loaded local dataset from: {LOCAL_DATASET_PATH} ({len(dataset)} samples)")
else:
    dataset = load_dataset(DATASET_NAME, split=SPLIT)
    print(f"Loaded split '{SPLIT}' from Hugging Face with {len(dataset)} samples")

def extract_gloss(sample):
    if "label" in sample and sample["label"] is not None:
        return str(sample["label"]).strip().lower()
    gloss = sample.get("gloss")
    if isinstance(gloss, dict):
        for key in ("label", "gloss", "text", "name"):
            if key in gloss and gloss[key] is not None:
                return str(gloss[key]).strip().lower()
    return str(gloss).strip().lower() if gloss is not None else "unknown"

Rate limited. Waiting 107.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_19/11105.mp4
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_19/11113.mp4
Rate limited. Waiting 107.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_19/11143.mp4
Rate limited. Waiting 107.0s before retry [Retry 1/5].
Rate limited. Waiting 107.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf5e6fd0/data/data_19/11148.mp4
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/Voxel51/WLASL/resolve/3cf8daaac08088798f539d62fa511028bf

Loaded split 'train' with 11880 samples


In [22]:
"""# 2. Exploratory Data Analysis"""

label_only_dataset = dataset.remove_columns([col for col in dataset.column_names if col != "label"])
raw_labels = [str(sample["label"]).strip().lower() for sample in label_only_dataset]
class_counts = Counter(raw_labels)

print(f"Total labels retrieved: {len(raw_labels)}")
print(f"Unique Labels: {len(class_counts)}")

print("\nTop 20 most common signs:")
for label, count in class_counts.most_common(20):
    print(f"  {label}: {count} samples")

print("\nBottom 20 least common signs:")
for label, count in class_counts.most_common()[-20:]:
    print(f"  {label}: {count} samples")

print(f"\nAvg samples per class: {len(raw_labels) / max(1, len(class_counts)):.2f}")

Total labels retrieved: 11880
Unique Labels: 119

Top 20 most common signs:
  0: 100 samples
  1: 100 samples
  2: 100 samples
  3: 100 samples
  4: 100 samples
  5: 100 samples
  6: 100 samples
  7: 100 samples
  8: 100 samples
  9: 100 samples
  10: 100 samples
  11: 100 samples
  12: 100 samples
  13: 100 samples
  14: 100 samples
  15: 100 samples
  16: 100 samples
  17: 100 samples
  18: 100 samples
  19: 100 samples

Bottom 20 least common signs:
  100: 100 samples
  101: 100 samples
  102: 100 samples
  103: 100 samples
  104: 100 samples
  105: 100 samples
  106: 100 samples
  107: 100 samples
  108: 100 samples
  109: 100 samples
  110: 100 samples
  111: 100 samples
  112: 100 samples
  113: 100 samples
  114: 100 samples
  115: 100 samples
  116: 100 samples
  117: 100 samples
  118: 100 samples
  23: 80 samples

Avg samples per class: 99.83


In [23]:
# WLASL-100 sanity check
top_k_labels = [label for label, _ in class_counts.most_common(TOP_K)]
top_k_set = set(top_k_labels)

filtered_labels = [lbl for lbl in raw_labels if lbl in top_k_set]
filtered_counts = Counter(filtered_labels)

print(f"Top-{TOP_K} classes selected: {len(top_k_labels)}")
print(f"Samples belonging to top-{TOP_K}: {len(filtered_labels)}")

print("\nTop 10 classes within selected subset:")
for label, count in filtered_counts.most_common(10):
    print(f"  {label}: {count}")

out_dir = Path("processed/wlasl_top100")
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / "labels_preview.txt").write_text("\n".join(top_k_labels), encoding="utf-8")
print(f"Saved preview labels to: {out_dir / 'labels_preview.txt'}")

Top-100 classes selected: 100
Samples belonging to top-100: 10000

Top 10 classes within selected subset:
  0: 100
  1: 100
  2: 100
  3: 100
  4: 100
  5: 100
  6: 100
  7: 100
  8: 100
  9: 100
Saved preview labels to: processed/wlasl_top100/labels_preview.txt


In [27]:
# Save the full dataset under wlasl_full_dataset folder
full_dataset_dir = Path("wlasl_full_dataset")
full_dataset_dir.mkdir(parents=True, exist_ok=True)

# Save dataset information (Arrow files)
dataset.save_to_disk(str(full_dataset_dir / "dataset"))

# Reuse already computed labels if available; otherwise build from label-only dataset
if "raw_labels" in globals():
    all_labels = raw_labels
else:
    label_only_dataset = dataset.remove_columns(
        [col for col in dataset.column_names if col != "label"]
    )
    all_labels = [str(x).strip().lower() for x in label_only_dataset["label"]]

(full_dataset_dir / "all_labels.txt").write_text("\n".join(all_labels), encoding="utf-8")

# Recompute class counts if needed
if "class_counts" not in globals():
    class_counts = Counter(all_labels)

class_stats = "\n".join([f"{label}: {count}" for label, count in class_counts.most_common()])
(full_dataset_dir / "class_statistics.txt").write_text(class_stats, encoding="utf-8")

print(f"Full dataset saved to: {full_dataset_dir}")
print(f"Total samples: {len(dataset)}")
print(f"Total unique classes: {len(class_counts)}")

Saving the dataset (10/10 shards): 100%|██████████| 11880/11880 [00:21<00:00, 562.20 examples/s]

Full dataset saved to: wlasl_full_dataset
Total samples: 11880
Total unique classes: 119
